# OrbitRisk Vision — Treinamento e Avaliação de CNNs

**Global Solution — Computer Vision**  
**Integrante:** Victoria Franceschini Pizza — RM550609

Este notebook documenta o desenvolvimento do projeto **OrbitRisk Vision**, uma solução de Visão Computacional aplicada ao contexto da Indústria Espacial.

O objetivo é classificar imagens satelitais em cinco categorias de uso do solo:

- `forest` — Floresta / Vegetação
- `industrial` — Área Industrial
- `residential` — Área Residencial
- `river` — Rio
- `sealake` — Mar ou Lago


## 1. Definição do problema

O problema tratado neste projeto é uma tarefa de **classificação supervisionada de imagens**.

A entrada do modelo é uma imagem satelital, e a saída esperada é a classe correspondente ao tipo de área presente na imagem.

Essa solução se conecta à **Indústria Espacial** porque utiliza imagens de satélite para análise automática do território, podendo apoiar monitoramento ambiental, análise urbana e tomada de decisão baseada em sensoriamento remoto.


## 2. Importação das bibliotecas

Nesta etapa são importadas as bibliotecas necessárias para manipulação dos dados, criação das CNNs, treinamento, avaliação e visualização dos resultados.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


## 3. Configurações iniciais

O dataset foi organizado em três conjuntos: treino, validação e teste.

A divisão utilizada foi:

| Conjunto | Imagens por classe | Total |
|---|---:|---:|
| Treino | 200 | 1000 |
| Validação | 40 | 200 |
| Teste | 40 | 200 |

As imagens foram redimensionadas para **128x128 pixels**.


In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 30

TRAIN_DIR = "../dataset/train"
VAL_DIR = "../dataset/val"
TEST_DIR = "../dataset/test"

MODELS_DIR = "../models"
RESULTS_DIR = "../results"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


## 4. Carregamento e pré-processamento das imagens

As imagens foram normalizadas para valores entre 0 e 1.

No conjunto de treino, foram aplicadas técnicas de **data augmentation**, como rotação, zoom, deslocamento e espelhamento horizontal. Isso ajuda a melhorar a generalização do modelo.


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1.0 / 255)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
class_names


## 5. Arquitetura 1 — CNN Simples

A primeira CNN foi criada como modelo base. Ela possui uma estrutura mais simples, com camadas convolucionais, pooling, camada densa e dropout.


In [ ]:
def criar_cnn_simples(input_shape=(128, 128, 3), num_classes=5):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),

        layers.Dense(num_classes, activation="softmax")
    ])

    return model


cnn_simples = criar_cnn_simples()
cnn_simples.summary()


## 6. Arquitetura 2 — CNN Profunda

A segunda CNN possui mais camadas convolucionais e maior capacidade de aprendizado. Também utiliza regularização L2 e Dropout para reduzir o risco de overfitting.


In [ ]:
def criar_cnn_profunda(input_shape=(128, 128, 3), num_classes=5):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),

        layers.Dense(
            256,
            activation="relu",
            kernel_regularizer=regularizers.l2(0.001)
        ),
        layers.Dropout(0.4),

        layers.Dense(
            128,
            activation="relu",
            kernel_regularizer=regularizers.l2(0.001)
        ),
        layers.Dropout(0.3),

        layers.Dense(num_classes, activation="softmax")
    ])

    return model


cnn_profunda = criar_cnn_profunda()
cnn_profunda.summary()


## 7. Funções auxiliares de treinamento e gráficos

A função abaixo treina o modelo e salva os gráficos de acurácia e loss na pasta `results`.


In [ ]:
def plotar_historico(history, nome_modelo):
    plt.figure()
    plt.plot(history.history["accuracy"], label="Treino")
    plt.plot(history.history["val_accuracy"], label="Validação")
    plt.title(f"Acurácia - {nome_modelo}")
    plt.xlabel("Épocas")
    plt.ylabel("Acurácia")
    plt.legend()
    plt.savefig(os.path.join(RESULTS_DIR, f"accuracy_{nome_modelo}.png"))
    plt.show()

    plt.figure()
    plt.plot(history.history["loss"], label="Treino")
    plt.plot(history.history["val_loss"], label="Validação")
    plt.title(f"Loss - {nome_modelo}")
    plt.xlabel("Épocas")
    plt.ylabel("Loss")
    plt.legend()
    plt.savefig(os.path.join(RESULTS_DIR, f"loss_{nome_modelo}.png"))
    plt.show()


def treinar_modelo(modelo, nome_modelo):
    modelo.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),
        ModelCheckpoint(
            filepath=os.path.join(MODELS_DIR, f"{nome_modelo}.keras"),
            monitor="val_accuracy",
            save_best_only=True
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=3,
            min_lr=0.00001
        )
    ]

    history = modelo.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS,
        callbacks=callbacks
    )

    plotar_historico(history, nome_modelo)

    return history


## 8. Treinamento da CNN Simples

Esta célula treina a primeira arquitetura. Caso o modelo já tenha sido treinado pelo script `src/train.py`, esta etapa pode ser usada apenas para documentação.


In [ ]:
history_simples = treinar_modelo(cnn_simples, "cnn_simples")


## 9. Treinamento da CNN Profunda

Esta célula treina a segunda arquitetura. A CNN Profunda foi a arquitetura com melhor desempenho no projeto.


In [ ]:
history_profunda = treinar_modelo(cnn_profunda, "cnn_profunda")


## 10. Avaliação dos modelos

Nesta etapa, os modelos são avaliados no conjunto de teste. São exibidas acurácia, loss, relatório de classificação e matriz de confusão.


In [ ]:
def avaliar_modelo(nome_modelo):
    caminho_modelo = os.path.join(MODELS_DIR, f"{nome_modelo}.keras")
    modelo = tf.keras.models.load_model(caminho_modelo)

    print(f"\nAvaliando modelo: {nome_modelo}")

    loss, accuracy = modelo.evaluate(test_generator)

    print(f"Loss no teste: {loss:.4f}")
    print(f"Acurácia no teste: {accuracy:.4f}")

    probabilidades = modelo.predict(test_generator)
    y_pred = np.argmax(probabilidades, axis=1)
    y_true = test_generator.classes

    print("\nRelatório de classificação:")
    print(classification_report(y_true, y_pred, target_names=class_names))

    matriz = confusion_matrix(y_true, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=class_names
    )

    plt.figure(figsize=(8, 6))
    disp.plot(values_format="d")
    plt.title(f"Matriz de Confusão - {nome_modelo}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"matriz_confusao_{nome_modelo}.png"))
    plt.show()


avaliar_modelo("cnn_simples")
avaliar_modelo("cnn_profunda")


## 11. Resultados obtidos

Os resultados finais obtidos no conjunto de teste foram:

| Modelo | Acurácia no teste | Loss no teste |
|---|---:|---:|
| CNN Simples | 89% | 0.2756 |
| CNN Profunda | 97% | 0.2226 |

A CNN Profunda apresentou o melhor desempenho, superando a referência mínima de 88% de acurácia.


## 12. Teste com imagem nova

Nesta etapa, o melhor modelo treinado é carregado e utilizado para classificar uma imagem nova do conjunto de teste.


In [ ]:
from tensorflow.keras.preprocessing import image

MODEL_PATH = os.path.join(MODELS_DIR, "cnn_profunda.keras")
modelo = tf.keras.models.load_model(MODEL_PATH)

CLASS_NAMES = [
    "forest",
    "industrial",
    "residential",
    "river",
    "sealake"
]


def prever_imagem(caminho_imagem):
    img = image.load_img(caminho_imagem, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predicoes = modelo.predict(img_array)

    indice_classe = np.argmax(predicoes[0])
    classe_prevista = CLASS_NAMES[indice_classe]
    confianca = predicoes[0][indice_classe] * 100

    return classe_prevista, confianca, predicoes[0]


caminho_teste = "../dataset/test/forest"
primeira_imagem = os.listdir(caminho_teste)[0]
caminho_completo = os.path.join(caminho_teste, primeira_imagem)

classe, confianca, probabilidades = prever_imagem(caminho_completo)

print(f"Imagem testada: {caminho_completo}")
print(f"Classe prevista: {classe}")
print(f"Confiança: {confianca:.2f}%")
print("Probabilidades:", probabilidades)


## 13. Conclusão

O projeto demonstrou a aplicação prática de redes neurais convolucionais no contexto da Indústria Espacial.

Foram criadas duas CNNs do zero, sem uso de modelos pré-treinados. Os modelos foram treinados, avaliados e comparados usando métricas quantitativas, gráficos de acurácia/loss e matriz de confusão.

A CNN Profunda apresentou o melhor desempenho, alcançando **97% de acurácia no conjunto de teste**, superando a meta mínima de 88%.

Além disso, foi criada uma aplicação em Streamlit para demonstrar o modelo funcionando com novas imagens.
